# Marketing Channel ROI Comparison
## Lab: Statistical Analysis for Budget Allocation

**Scenario:** As a marketing analyst at a growing e-commerce company, the CMO
has requested a statistically rigorous analysis of marketing channel performance
to guide the allocation of a $500K monthly budget across 5 channels.

**Dataset:** Nykaa Marketing Campaign Performance Dataset (Kaggle)
A structured, real-world inspired e-commerce marketing campaign dataset designed
for ROI analysis, performance optimization, and predictive modeling.

**Analyst:** Marco Martins
**Date:** April 2026

## Part 1: Dataset Discovery & Exploration

### Step 1: Dataset Selection

After evaluating multiple Kaggle datasets for suitability, the **Nykaa Marketing
Campaign Performance Dataset** was selected for the following reasons:

- 55,555 campaign records across 5 marketing channels
- Contains all required metrics: CPA, ROI, Conversions, Revenue, Clicks, Impressions
- ~11,000 rows per channel — sufficient for robust statistical testing
- Clean data with no missing values
- Real-world inspired structure matching the lab scenario

### Step 2: Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, fisher_exact, false_discovery_control

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("✅ Libraries loaded successfully")

### Step 3: Load & Inspect the Dataset

We load the Nykaa dataset and perform an initial inspection to understand
its structure, data types, and quality before any analysis.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Load & Inspect
# ═══════════════════════════════════════════════════════════════════
# Adjust filename if yours is different
df = pd.read_csv("nykaa_campaign_data.csv")

print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"Shape:        {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("\nColumn names & data types:")
print(df.dtypes)

print("\nFirst 3 rows:")
print(df.head(3))

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nCampaign_Type (our grouping variable):")
print(df["Campaign_Type"].value_counts())

### Step 4: Data Cleaning & Feature Engineering

The dataset is already clean (no missing values). We perform the following
transformations:

- **Parse dates** so we can do time-based analysis if needed
- **Calculate CTR** (Click-Through Rate = Clicks / Impressions)
- **Calculate ROAS** (Return on Ad Spend = Revenue / Spend), where
  Spend = Acquisition_Cost × Conversions
- **Calculate CPA** directly from Acquisition_Cost (already provided)
- **Calculate Conversion Rate** = Conversions / Clicks

These derived metrics will be the basis for our statistical comparisons.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Clean & Engineer Features
# ═══════════════════════════════════════════════════════════════════

# Parse dates (format is DD-MM-YYYY)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

# Derived metrics
df["CTR"]          = df["Clicks"]      / df["Impressions"]   # Click-Through Rate
df["Conv_Rate"]    = df["Conversions"] / df["Clicks"]        # Conversion Rate
df["Spend"]        = df["Acquisition_Cost"] * df["Conversions"]  # Total Spend
df["ROAS"]         = df["Revenue"]     / df["Spend"]         # Return on Ad Spend
df["Non_Conversions"] = df["Clicks"]   - df["Conversions"]   # For Fisher's test later

# Handle any infinite values from division by zero
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# ── Verify ────────────────────────────────────────────────────────
print("=" * 55)
print("ENGINEERED FEATURES — SAMPLE VALUES")
print("=" * 55)
print(df[["Campaign_Type", "CTR", "Conv_Rate",
          "Spend", "ROAS", "Acquisition_Cost"]].head(5).round(4))

print(f"\nInfinite/NaN values after cleaning:")
print(df[["CTR", "Conv_Rate", "Spend", "ROAS"]].isnull().sum())

print(f"\nDate range: {df['Date'].min().date()} → {df['Date'].max().date()}")

# Save cleaned dataset
df.to_csv("marketing_data.csv", index=False)
print("\n✅ Cleaned dataset saved as marketing_data.csv")

### Step 5: Aggregate Metrics by Channel

We calculate key marketing KPIs for each campaign type:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| CPA | Acquisition_Cost (given) | Lower is better — cost to acquire one customer |
| ROAS | Revenue / Spend | Higher is better — revenue per £1 spent |
| Conv Rate | Conversions / Clicks | Higher is better — % of clicks that convert |
| CTR | Clicks / Impressions | Higher is better — % of impressions that get clicked |
| ROI | Given directly | Higher is better — overall profitability |

These metrics will be compared statistically in Part 2.

In [ ]:
#═══════════════════════════════════════════════════════════════════
# Aggregate Metrics Table
# ═══════════════════════════════════════════════════════════════════

agg = df.groupby("Campaign_Type").agg(
    N                = ("Campaign_ID",       "count"),
    Avg_CPA          = ("Acquisition_Cost",  "mean"),
    Std_CPA          = ("Acquisition_Cost",  "std"),
    Avg_ROAS         = ("ROAS",              "mean"),
    Avg_ROI          = ("ROI",               "mean"),
    Avg_Conv_Rate    = ("Conv_Rate",         "mean"),
    Avg_CTR          = ("CTR",               "mean"),
    Avg_Engagement   = ("Engagement_Score",  "mean"),
    Total_Revenue    = ("Revenue",           "sum"),
    Total_Spend      = ("Spend",             "sum"),
    Total_Conversions= ("Conversions",       "sum"),
    Total_Clicks     = ("Clicks",            "sum"),
    Total_Non_Conv   = ("Non_Conversions",   "sum"),
).reset_index()

# Display summary table
print("=" * 75)
print("CHANNEL PERFORMANCE SUMMARY TABLE")
print("=" * 75)
display_cols = agg[["Campaign_Type", "N", "Avg_CPA", "Avg_ROAS",
                     "Avg_ROI", "Avg_Conv_Rate", "Avg_CTR", "Avg_Engagement"]].copy()
display_cols.columns = ["Channel", "N", "Avg CPA", "Avg ROAS",
                        "Avg ROI", "Avg Conv Rate", "Avg CTR", "Avg Engagement"]
display_cols = display_cols.round(4)
print(display_cols.to_string(index=False))

print("\n── Variability (Std Dev of CPA) ──────────────────────────────────")
print(agg[["Campaign_Type", "Std_CPA"]].round(2).to_string(index=False))

print("\n── Total Revenue & Spend by Channel ──────────────────────────────")
rev_spend = agg[["Campaign_Type", "Total_Revenue", "Total_Spend",
                  "Total_Conversions"]].copy()
rev_spend["Profit"] = rev_spend["Total_Revenue"] - rev_spend["Total_Spend"]
print(rev_spend.round(0).to_string(index=False))

### Step 6: Visualizing Channel Performance

We create two sets of visualizations:

1. **Bar charts** — aggregate KPIs per channel (overview)
2. **Distribution plots** — box plots and histograms showing the spread
   of CPA, ROAS and Conversion Rate per channel

Distribution plots are critical because they reveal whether channels
truly differ or whether high variance makes differences unreliable.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Bar Chart Overview
# ═══════════════════════════════════════════════════════════════════

channels = agg["Campaign_Type"]
colors   = sns.color_palette("husl", len(channels))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Marketing Channel Performance Overview", fontsize=16, fontweight="bold")

# 1. Avg CPA
ax = axes[0, 0]
bars = ax.barh(channels, agg["Avg_CPA"], color=colors)
ax.set_xlabel("Avg CPA ($)")
ax.set_title("Avg Cost Per Acquisition")
for bar, val in zip(bars, agg["Avg_CPA"]):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f"${val:,.0f}", va="center", fontsize=9)

# 2. Avg ROAS
ax = axes[0, 1]
bars = ax.barh(channels, agg["Avg_ROAS"], color=colors)
ax.set_xlabel("Avg ROAS")
ax.set_title("Avg Return on Ad Spend")
for bar, val in zip(bars, agg["Avg_ROAS"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}", va="center", fontsize=9)

# 3. Avg Conversion Rate
ax = axes[0, 2]
bars = ax.barh(channels, agg["Avg_Conv_Rate"] * 100, color=colors)
ax.set_xlabel("Avg Conversion Rate (%)")
ax.set_title("Avg Conversion Rate")
for bar, val in zip(bars, agg["Avg_Conv_Rate"] * 100):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}%", va="center", fontsize=9)

# 4. Total Conversions
ax = axes[1, 0]
bars = ax.barh(channels, agg["Total_Conversions"], color=colors)
ax.set_xlabel("Total Conversions")
ax.set_title("Total Conversions by Channel")
for bar, val in zip(bars, agg["Total_Conversions"]):
    ax.text(val + 1000, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9)

# 5. Avg CTR
ax = axes[1, 1]
bars = ax.barh(channels, agg["Avg_CTR"] * 100, color=colors)
ax.set_xlabel("Avg CTR (%)")
ax.set_title("Avg Click-Through Rate")
for bar, val in zip(bars, agg["Avg_CTR"] * 100):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}%", va="center", fontsize=9)

# 6. Avg ROI
ax = axes[1, 2]
bars = ax.barh(channels, agg["Avg_ROI"], color=colors)
ax.set_xlabel("Avg ROI")
ax.set_title("Avg Return on Investment")
for bar, val in zip(bars, agg["Avg_ROI"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("group_metrics_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved group_metrics_overview.png")

# ═══════════════════════════════════════════════════════════════════
# Distribution Plots
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Metric Distributions by Channel", fontsize=15, fontweight="bold")

metrics = [
    ("Acquisition_Cost", "CPA ($)",          axes[0]),
    ("ROAS",             "ROAS",             axes[1]),
    ("Conv_Rate",        "Conversion Rate",  axes[2]),
]

for col, label, ax in metrics:
    data = [df[df["Campaign_Type"] == ch][col].dropna() for ch in channels]
    bp = ax.boxplot(data, patch_artist=True, notch=True,
                    medianprops=dict(color="black", linewidth=2))
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels(channels, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel(label)
    ax.set_title(f"Distribution of {label}")

plt.tight_layout()
plt.savefig("group_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved group_distributions.png")

## Part 1 Observations: Dataset Exploration & Key Insights

### Dataset Summary
- **Source:** Nykaa Marketing Campaign Performance Dataset (Kaggle)
- **Size:** 55,555 campaigns across 5 channels (~11,000 per channel)
- **Period:** July 2024 – June 2025 (12 months)
- **Metrics available:** CPA, ROAS, ROI, Conversion Rate, CTR, Engagement Score

---

### 1. Which groups perform best on each metric?

| Metric | Best Channel | Worst Channel |
|--------|-------------|---------------|
| CPA (lower is better) | **Social Media** ($373) | **Email** ($380) |
| ROAS (higher is better) | **Social Media** (3.75) | **Email** (3.68) |
| ROI (higher is better) | **Social Media** (2.75) | **Email** (2.68) |
| Conversion Rate | **Email** (22.08%) | **Paid Ads** (21.81%) |
| CTR | **Influencer** (8.56%) | **Email** (8.47%) |
| Total Conversions | **Social Media** (11.6M) | **SEO** (11.4M) |

> **Social Media consistently outperforms** on cost and return metrics, while
> **Email underperforms** on CPA, ROAS and ROI despite having the highest
> conversion rate — suggesting Email drives conversions but at higher cost.

---

### 2. Which groups show the most variability?

- **Paid Ads** has the highest CPA standard deviation ($566) — the most
  unpredictable channel, with some campaigns costing over **$15,000** per
  acquisition
- **ROAS** is highly variable across all channels, with extreme outliers
  reaching up to **70x return** — likely driven by viral or seasonal campaigns
- **Conversion Rate** is the most stable metric across all channels,
  with interquartile ranges largely overlapping

---

### 3. Are there any obvious outliers?

- **CPA outliers:** Values up to ~$15,000 observed in Paid Ads (vs. median
  of ~$200), indicating a small number of extremely costly campaigns
- **ROAS outliers:** Values up to ~70x in multiple channels, pulling means
  upward significantly
- All distributions are **heavily right-skewed** — the bulk of campaigns
  cluster at low cost/moderate return, but a long tail of extreme values
  inflates the averages

---

### 4. What patterns do we observe?

- **Aggregate differences between channels are small** (e.g., CPA ranges
  only $7 from best to worst), but with ~11,000 observations per channel,
  even tiny differences may reach statistical significance
- The **large within-channel variance** (CPA std ~$520–$566) relative to
  between-channel differences raises an important question: are the observed
  differences real or just noise?
- This motivates the **statistical testing in Part 2** — we cannot rely on
  visual inspection alone when variance is this high

---

> ⚠️ **Important caveat:** Statistical significance ≠ practical significance.
> A $7 difference in average CPA may be statistically detectable with 11,000
> samples but may not justify a major budget reallocation. 

## Part 2: Pairwise Group Comparisons

### Step 4: Compare Channels Using Independent t-tests

We compare **CPA** and **ROAS** between every pair of channels using
independent two-sample t-tests.

**Why t-tests?**
CPA and ROAS are continuous metrics — the t-test is appropriate for
comparing means of two independent groups.

**For each pair we calculate:**
- **t-statistic & p-value** — is the difference statistically significant?
- **Cohen's d** — is the difference *practically* significant?
  - |d| < 0.2 → negligible
  - |d| < 0.5 → small
  - |d| < 0.8 → medium
  - |d| ≥ 0.8 → large

With 5 channels there are **10 unique pairs** per metric = 20 total comparisons.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Pairwise t-tests for CPA and ROAS
# ═══════════════════════════════════════════════════════════════════
from itertools import combinations

def cohens_d(a, b):
    """Calculate Cohen's d effect size between two arrays."""
    pooled_std = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
    return (np.mean(b) - np.mean(a)) / pooled_std if pooled_std > 0 else 0.0

def interpret_d(d):
    """Interpret Cohen's d magnitude."""
    d = abs(d)
    if d < 0.2:   return "negligible"
    elif d < 0.5: return "small"
    elif d < 0.8: return "medium"
    else:         return "large"

def run_pairwise_ttests(df, metric):
    """Run all pairwise t-tests for a given metric across channels."""
    # Clean data
    data = df[["Campaign_Type", metric]].dropna()
    data = data[~np.isinf(data[metric])]
    channel_list = sorted(data["Campaign_Type"].unique())

    results = []
    for ch_a, ch_b in combinations(channel_list, 2):
        vals_a = data[data["Campaign_Type"] == ch_a][metric].values
        vals_b = data[data["Campaign_Type"] == ch_b][metric].values

        t_stat, p_val = ttest_ind(vals_a, vals_b)
        mean_a, mean_b = np.mean(vals_a), np.mean(vals_b)
        diff           = mean_b - mean_a
        pct_diff       = (diff / mean_a) * 100
        d              = cohens_d(vals_a, vals_b)

        results.append({
            "Channel A":    ch_a,
            "Channel B":    ch_b,
            "Mean A":       round(mean_a, 4),
            "Mean B":       round(mean_b, 4),
            "Difference":   round(diff, 4),
            "% Difference": round(pct_diff, 2),
            "t-statistic":  round(t_stat, 4),
            "p-value":      round(p_val, 6),
            "Cohen's d":    round(d, 4),
            "Effect Size":  interpret_d(d),
            "Significant":  p_val < 0.05,
        })

    return pd.DataFrame(results)

# ── Run for CPA ───────────────────────────────────────────────────
print("=" * 70)
print("PAIRWISE T-TESTS: CPA (Acquisition_Cost)")
print("=" * 70)
cpa_results = run_pairwise_ttests(df, "Acquisition_Cost")
print(cpa_results.to_string(index=False))
print(f"\nTotal comparisons: {len(cpa_results)}")
print(f"Significant at α=0.05: {cpa_results['Significant'].sum()}")

# ── Run for ROAS ──────────────────────────────────────────────────
print("\n" + "=" * 70)
print("PAIRWISE T-TESTS: ROAS")
print("=" * 70)
roas_results = run_pairwise_ttests(df, "ROAS")
print(roas_results.to_string(index=False))
print(f"\nTotal comparisons: {len(roas_results)}")
print(f"Significant at α=0.05: {roas_results['Significant'].sum()}")

### Visualizing Pairwise Comparisons: P-value Heatmaps

A p-value heatmap shows at a glance which channel pairs are
statistically different. 

- **Red** = low p-value (significant difference)
- **Green** = high p-value (no significant difference)

Values above 0.05 indicate we cannot reject the null hypothesis
that the two channels perform equally.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P-value Heatmaps
# ═══════════════════════════════════════════════════════════════════

def plot_pvalue_heatmap(results, metric_name, filename):
    """Build a symmetric p-value matrix and plot as heatmap."""
    channels = sorted(set(results["Channel A"]) | set(results["Channel B"]))
    n = len(channels)
    idx = {ch: i for i, ch in enumerate(channels)}

    # Build symmetric matrix (diagonal = 1.0)
    matrix = np.ones((n, n))
    for _, row in results.iterrows():
        i, j = idx[row["Channel A"]], idx[row["Channel B"]]
        matrix[i, j] = row["p-value"]
        matrix[j, i] = row["p-value"]

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        matrix,
        xticklabels=channels,
        yticklabels=channels,
        annot=True, fmt=".3f",
        cmap="RdYlGn",
        vmin=0, vmax=1,
        linewidths=0.5,
        ax=ax,
        cbar_kws={"label": "p-value"}
    )
    ax.set_title(f"Pairwise T-test P-values: {metric_name}\n"
                 f"(Green = no significant difference, Red = significant)",
                 fontsize=11)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved {filename}")

plot_pvalue_heatmap(cpa_results,  "CPA (Acquisition Cost)", "cpa_comparison_heatmap.png")
plot_pvalue_heatmap(roas_results, "ROAS",                   "roas_comparison_heatmap.png")

### Step 5: Compare Binary Outcomes Using Fisher's Exact Test

While t-tests compare continuous metrics (CPA, ROAS), Fisher's exact test
compares **binary outcomes** — specifically whether a click resulted in a
conversion or not.

**Why Fisher's exact test?**
As shown in the teacher's notebook (03_statistical_analysis.ipynb), when
our outcome is binary (converted / not converted), the data follows a
Binomial distribution — Fisher's exact test is the appropriate choice,
not a t-test.

**Setup:** For each channel pair we build a 2×2 contingency table:

|          | Converted | Not Converted |
|----------|-----------|---------------|
| Channel A | conv_A   | non_conv_A    |
| Channel B | conv_B   | non_conv_B    |

Where: Non_Conversions = Clicks - Conversions

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Fisher's Exact Test
# ═══════════════════════════════════════════════════════════════════

# Aggregate totals per channel for Fisher's test
fisher_agg = df.groupby("Campaign_Type").agg(
    Total_Conversions    = ("Conversions",     "sum"),
    Total_Clicks         = ("Clicks",          "sum"),
    Total_Non_Conversions= ("Non_Conversions", "sum"),
).reset_index()

fisher_agg["Conv_Rate"] = (fisher_agg["Total_Conversions"] /
                           fisher_agg["Total_Clicks"]).round(4)

print("=" * 65)
print("CONVERSION SUMMARY BY CHANNEL")
print("=" * 65)
print(fisher_agg[["Campaign_Type", "Total_Conversions",
                   "Total_Clicks", "Total_Non_Conversions",
                   "Conv_Rate"]].to_string(index=False))

# ── Pairwise Fisher's Exact Tests ─────────────────────────────────
fisher_results = []
channel_list = sorted(fisher_agg["Campaign_Type"].unique())

for ch_a, ch_b in combinations(channel_list, 2):
    row_a = fisher_agg[fisher_agg["Campaign_Type"] == ch_a].iloc[0]
    row_b = fisher_agg[fisher_agg["Campaign_Type"] == ch_b].iloc[0]

    table = [
        [row_a["Total_Conversions"], row_a["Total_Non_Conversions"]],
        [row_b["Total_Conversions"], row_b["Total_Non_Conversions"]],
    ]

    odds_ratio, p_val = fisher_exact(table, alternative="two-sided")
    rate_a = row_a["Conv_Rate"]
    rate_b = row_b["Conv_Rate"]
    diff   = rate_b - rate_a
    pct_diff = (diff / rate_a) * 100

    fisher_results.append({
        "Channel A":    ch_a,
        "Channel B":    ch_b,
        "Rate A":       round(rate_a, 4),
        "Rate B":       round(rate_b, 4),
        "Difference":   round(diff, 4),
        "% Difference": round(pct_diff, 2),
        "Odds Ratio":   round(odds_ratio, 4),
        "p-value":      round(p_val, 6),
        "Significant":  p_val < 0.05,
    })

fisher_df = pd.DataFrame(fisher_results)

print("\n" + "=" * 65)
print("FISHER'S EXACT TEST: CONVERSION RATES")
print("=" * 65)
print(fisher_df.to_string(index=False))
print(f"\nTotal comparisons: {len(fisher_df)}")
print(f"Significant at α=0.05: {fisher_df['Significant'].sum()}")

# ── Bar chart of conversion rates ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors = sns.color_palette("husl", len(fisher_agg))
bars = ax.barh(fisher_agg["Campaign_Type"],
               fisher_agg["Conv_Rate"] * 100, color=colors)
for bar, val in zip(bars, fisher_agg["Conv_Rate"] * 100):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}%", va="center", fontsize=10)
ax.set_xlabel("Conversion Rate (%)")
ax.set_title("Conversion Rate by Channel\n(Conversions / Clicks)")
plt.tight_layout()
plt.savefig("rate_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved rate_comparison.png")

## Part 3: Multiple Comparisons Correction

### Step 6: Applying Bonferroni and Benjamini-Hochberg Corrections

**The multiple comparisons problem:**
When we run many hypothesis tests simultaneously, we inflate the probability
of getting false positives purely by chance. With α=0.05 and 20 tests, we
expect 1 false positive even if all channels are truly identical.

We ran **20 total comparisons** (10 CPA + 10 conversion rate):
- Expected false positives at α=0.05: 20 × 0.05 = **1.0**

**Two correction methods:**

| Method | Controls | Conservative? |
|--------|---------|--------------|
| **Bonferroni** | Family-wise error rate (P(any false positive)) | Very conservative |
| **Benjamini-Hochberg (FDR)** | Expected proportion of false discoveries | Less conservative |

As shown in the teacher's notebook, FDR is preferred when testing many
hypotheses — it sacrifices less statistical power than Bonferroni.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Multiple Comparisons Correction
# ═══════════════════════════════════════════════════════════════════

alpha = 0.05
n_cpa     = len(cpa_results)      # 10 comparisons
n_fisher  = len(fisher_df)        # 10 comparisons
n_total   = n_cpa + n_fisher      # 20 total

print("=" * 60)
print("MULTIPLE COMPARISONS PROBLEM")
print("=" * 60)
print(f"CPA comparisons:              {n_cpa}")
print(f"Conversion rate comparisons:  {n_fisher}")
print(f"Total comparisons:            {n_total}")
print(f"Expected false positives:     {n_total * alpha:.1f}")
print(f"\nWithout correction, ~{n_total * alpha:.0f} 'significant' result(s)")
print("could be due to random chance alone.")

# ── Bonferroni Correction ─────────────────────────────────────────
alpha_bonf_cpa    = alpha / n_cpa
alpha_bonf_fisher = alpha / n_fisher

cpa_results["significant_bonferroni"] = (
    cpa_results["p-value"] < alpha_bonf_cpa)
fisher_df["significant_bonferroni"] = (
    fisher_df["p-value"] < alpha_bonf_fisher)

print("\n" + "=" * 60)
print("BONFERRONI CORRECTION")
print("=" * 60)
print(f"Adjusted α for CPA:             {alpha_bonf_cpa:.4f}")
print(f"Adjusted α for conversion rate: {alpha_bonf_fisher:.4f}")
print(f"\nCPA significant after Bonferroni:             "
      f"{cpa_results['significant_bonferroni'].sum()}/10")
print(f"Conversion rate significant after Bonferroni: "
      f"{fisher_df['significant_bonferroni'].sum()}/10")

# ── Benjamini-Hochberg FDR Correction ────────────────────────────
cpa_pvals    = cpa_results["p-value"].values
fisher_pvals = fisher_df["p-value"].values

cpa_results["p_value_fdr"] = false_discovery_control(cpa_pvals,    method="bh")
fisher_df["p_value_fdr"]   = false_discovery_control(fisher_pvals, method="bh")

cpa_results["significant_fdr"] = cpa_results["p_value_fdr"] < alpha
fisher_df["significant_fdr"]   = fisher_df["p_value_fdr"]   < alpha

print("\n" + "=" * 60)
print("BENJAMINI-HOCHBERG FDR CORRECTION")
print("=" * 60)
print(f"CPA significant after FDR:             "
      f"{cpa_results['significant_fdr'].sum()}/10")
print(f"Conversion rate significant after FDR: "
      f"{fisher_df['significant_fdr'].sum()}/10")

if fisher_df["significant_fdr"].sum() > 0:
    print("\nConversion rate pairs still significant after FDR:")
    sig = fisher_df[fisher_df["significant_fdr"]]
    print(sig[["Channel A", "Channel B", "Difference",
               "% Difference", "p-value", "p_value_fdr"]].to_string(index=False))

# ── Summary Comparison Table ──────────────────────────────────────
print("\n" + "=" * 60)
print("CORRECTION METHOD COMPARISON SUMMARY")
print("=" * 60)
summary = pd.DataFrame({
    "Method":          ["Uncorrected (α=0.05)", "Bonferroni", "FDR (BH)"],
    "CPA Significant": [
        cpa_results["Significant"].sum(),
        cpa_results["significant_bonferroni"].sum(),
        cpa_results["significant_fdr"].sum(),
    ],
    "Conv Rate Significant": [
        fisher_df["Significant"].sum(),
        fisher_df["significant_bonferroni"].sum(),
        fisher_df["significant_fdr"].sum(),
    ],
})
print(summary.to_string(index=False))

# ── Visualization ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(summary["Method"]))
w = 0.35
bars1 = ax.bar(x - w/2, summary["CPA Significant"],
               w, label="CPA", color="#2196F3", alpha=0.8)
bars2 = ax.bar(x + w/2, summary["Conv Rate Significant"],
               w, label="Conversion Rate", color="#FF9800", alpha=0.8)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha="center", fontsize=11)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha="center", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(summary["Method"], fontsize=11)
ax.set_ylabel("Number of Significant Comparisons")
ax.set_title("Effect of Multiple Comparisons Correction\non Significant Results")
ax.legend()
ax.set_ylim(0, 13)
plt.tight_layout()
plt.savefig("correction_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved correction_comparison.png")

## Part 2 Observations: Pairwise Group Comparisons

### T-test Results (CPA & ROAS)

**0 out of 10 pairs were statistically significant** for both CPA and ROAS.

| Metric | Significant Pairs | Most Different Pair | Difference |
|--------|------------------|---------------------|------------|
| CPA | 0/10 | Email vs Social Media | $6.87 (1.81%) |
| ROAS | 0/10 | Email vs Social Media | 0.072 (1.96%) |

**Key observations:**
- The largest CPA difference between any two channels is only **$6.87** — 
  on a base of ~$377, this is a 1.81% difference
- All Cohen's d values are **negligible (< 0.02)** — far below the 0.2
  threshold for even a "small" effect
- The heatmaps are **mostly green** but show some orange cells,
  particularly involving Social Media vs Email
- **Social Media vs Email** is consistently the "most different" pair
  across both metrics (p=0.33 for CPA, p=0.23 for ROAS) — visible as
  the lightest/most orange cells in both heatmaps — yet still far from
  the 0.05 significance threshold

> **Interpretation:** With ~11,000 observations per channel, the t-test has
> very high power. If real CPA or ROAS differences existed between channels,
> we would have detected them. The fact that we found nothing strongly
> suggests channels perform equivalently on these metrics.

---

### Fisher's Exact Test Results (Conversion Rate)

**10 out of 10 pairs were statistically significant** at α=0.05.

| Metric | Significant Pairs | Largest Difference | Smallest Difference |
|--------|------------------|--------------------|---------------------|
| Conversion Rate | 10/10 | Email vs Paid Ads (1.04%) | Influencer vs SEO (0.09%) |

**Key observations:**
- Every single channel pair shows a statistically significant difference
  in conversion rate
- However, the actual differences are **extremely small**: ranging from
  only **0.09% to 1.04%**
- The largest difference (Email 22.12% vs Paid Ads 21.89%) is just
  **0.23 percentage points**
- All odds ratios are very close to 1.0 (range: 0.989 – 1.013),
  confirming negligible practical difference

> ⚠️ **Critical insight — Statistical vs Practical Significance:**
> Fisher's test has enormous
> statistical power and detects differences as small as 0.09% as
> "significant." In practice, no marketing decision should be based
> on a 0.09% conversion rate difference — the business impact would
> be negligible.

---

### How do t-test and Fisher's results compare?

| Test | Metric | Significant? | Why? |
|------|--------|-------------|------|
| t-test | CPA | ❌ No | High variance ($520+ std) drowns out small mean differences |
| t-test | ROAS | ❌ No | Same — high within-channel variance |
| Fisher's | Conversion Rate | ✅ Yes (all) | 50M+ clicks gives enormous power to detect tiny differences |

The contrast between the two tests highlights an important principle:
**the same underlying reality (channels perform similarly) can produce
different statistical outcomes depending on sample size and test choice.**

---

### What does this mean for the CMO?

- On **cost efficiency** (CPA, ROAS): channels are statistically and
  practically equivalent — no basis for reallocation
- On **conversion rate**: channels are statistically different but
  practically equivalent — differences are too small to act on
- **The within-channel variance is the real story**: CPA standard
  deviations of $520–$566 dwarf the between-channel differences,
  suggesting **campaign-level optimization** matters far more than
  channel selection

## Part 3: Multiple Comparisons Correction

### Step 6: Applying Bonferroni and Benjamini-Hochberg Corrections

**The multiple comparisons problem:**
When we run many hypothesis tests simultaneously, we inflate the probability
of getting false positives purely by chance. With α=0.05 and 20 tests, we
expect 1 false positive even if all channels are truly identical.

We ran **20 total comparisons** (10 CPA t-tests + 10 Fisher's exact tests):
- Expected false positives at α=0.05: 20 × 0.05 = **1.0**

**Two correction methods:**

| Method | Controls | Conservative? |
|--------|---------|--------------|
| **Bonferroni** | Family-wise error rate — P(any false positive) | Very conservative |
| **Benjamini-Hochberg (FDR)** | Expected proportion of false discoveries | Less conservative |

As shown in the teacher's notebook (03_statistical_analysis.ipynb),
FDR is preferred when testing many hypotheses — it loses less
statistical power than Bonferroni while still controlling false discoveries.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Multiple Comparisons Correction
# ═══════════════════════════════════════════════════════════════════

alpha = 0.05
n_cpa    = len(cpa_results)   # 10 comparisons
n_fisher = len(fisher_df)     # 10 comparisons
n_total  = n_cpa + n_fisher   # 20 total

print("=" * 60)
print("MULTIPLE COMPARISONS PROBLEM")
print("=" * 60)
print(f"CPA comparisons:              {n_cpa}")
print(f"Conversion rate comparisons:  {n_fisher}")
print(f"Total comparisons:            {n_total}")
print(f"Expected false positives:     {n_total * alpha:.1f}")
print(f"\nWithout correction, ~{n_total * alpha:.0f} 'significant' result(s)")
print("could be due to random chance alone.")

# ── Bonferroni Correction ─────────────────────────────────────────
alpha_bonf_cpa    = alpha / n_cpa
alpha_bonf_fisher = alpha / n_fisher

cpa_results["significant_bonferroni"] = (
    cpa_results["p-value"] < alpha_bonf_cpa)
fisher_df["significant_bonferroni"] = (
    fisher_df["p-value"] < alpha_bonf_fisher)

print("\n" + "=" * 60)
print("BONFERRONI CORRECTION")
print("=" * 60)
print(f"Adjusted α for CPA:             {alpha_bonf_cpa:.4f}")
print(f"Adjusted α for conversion rate: {alpha_bonf_fisher:.4f}")
print(f"\nCPA significant after Bonferroni:             "
      f"{cpa_results['significant_bonferroni'].sum()}/10")
print(f"Conversion rate significant after Bonferroni: "
      f"{fisher_df['significant_bonferroni'].sum()}/10")

if fisher_df["significant_bonferroni"].sum() > 0:
    print("\nConversion rate pairs still significant after Bonferroni:")
    sig = fisher_df[fisher_df["significant_bonferroni"]]
    print(sig[["Channel A", "Channel B", "Difference",
               "% Difference", "p-value"]].to_string(index=False))

# ── Benjamini-Hochberg FDR Correction ────────────────────────────
cpa_pvals    = cpa_results["p-value"].values
fisher_pvals = fisher_df["p-value"].values

cpa_results["p_value_fdr"] = false_discovery_control(cpa_pvals,    method="bh")
fisher_df["p_value_fdr"]   = false_discovery_control(fisher_pvals, method="bh")

cpa_results["significant_fdr"] = cpa_results["p_value_fdr"] < alpha
fisher_df["significant_fdr"]   = fisher_df["p_value_fdr"]   < alpha

print("\n" + "=" * 60)
print("BENJAMINI-HOCHBERG FDR CORRECTION")
print("=" * 60)
print(f"CPA significant after FDR:             "
      f"{cpa_results['significant_fdr'].sum()}/10")
print(f"Conversion rate significant after FDR: "
      f"{fisher_df['significant_fdr'].sum()}/10")

if fisher_df["significant_fdr"].sum() > 0:
    print("\nConversion rate pairs still significant after FDR:")
    sig = fisher_df[fisher_df["significant_fdr"]]
    print(sig[["Channel A", "Channel B", "Difference",
               "% Difference", "p-value", "p_value_fdr"]].to_string(index=False))

# ── Summary Comparison Table ──────────────────────────────────────
print("\n" + "=" * 60)
print("CORRECTION METHOD COMPARISON SUMMARY")
print("=" * 60)
summary = pd.DataFrame({
    "Method": ["Uncorrected (α=0.05)", "Bonferroni", "FDR (BH)"],
    "CPA Significant": [
        int(cpa_results["Significant"].sum()),
        int(cpa_results["significant_bonferroni"].sum()),
        int(cpa_results["significant_fdr"].sum()),
    ],
    "Conv Rate Significant": [
        int(fisher_df["Significant"].sum()),
        int(fisher_df["significant_bonferroni"].sum()),
        int(fisher_df["significant_fdr"].sum()),
    ],
})
print(summary.to_string(index=False))

# ── Visualization ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(summary["Method"]))
w = 0.35
bars1 = ax.bar(x - w/2, summary["CPA Significant"],
               w, label="CPA", color="#2196F3", alpha=0.8)
bars2 = ax.bar(x + w/2, summary["Conv Rate Significant"],
               w, label="Conversion Rate", color="#FF9800", alpha=0.8)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha="center", fontsize=11)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha="center", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(summary["Method"], fontsize=11)
ax.set_ylabel("Number of Significant Comparisons")
ax.set_title("Effect of Multiple Comparisons Correction\non Number of Significant Results")
ax.legend()
ax.set_ylim(0, 13)
plt.tight_layout()
plt.savefig("correction_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved correction_comparison.png")

## Part 3 Observations: Multiple Comparisons Correction

### How many "significant" results were false positives?

With 20 total comparisons at α=0.05, we expected **1.0 false positive**
by chance alone — even if all channels were truly identical.

| Method | CPA Significant | Conv Rate Significant | Total |
|--------|----------------|----------------------|-------|
| Uncorrected (α=0.05) | 0/10 | 10/10 | 10/20 |
| Bonferroni | 0/10 | 9/10 | 9/20 |
| FDR (Benjamini-Hochberg) | 0/10 | 10/10 | 10/20 |

---

### CPA Results

All three methods agree: **0 significant CPA comparisons**.
This is consistent — CPA differences were already non-significant
before correction, and both methods confirm this finding.

---

### Conversion Rate Results

This is where the story gets interesting:

- **Uncorrected:** 10/10 significant — all pairs flagged
- **Bonferroni:** 9/10 significant — only **Influencer vs SEO** (the
  smallest difference at 0.09%) dropped out
- **FDR:** 10/10 significant — all pairs survive, including
  Influencer vs SEO (adjusted p = 0.0076)

Even the most conservative correction (Bonferroni) barely changes
the picture — 9 out of 10 pairs survive. This tells us the p-values
are so small (most showing as 0.000000) that no correction method
can rescue us from "significance."

---

### Which correction method is more appropriate here?

**FDR (Benjamini-Hochberg) is the better choice** for this scenario because:
- We are testing many hypotheses simultaneously (10 pairs)
- We care more about controlling the *proportion* of false discoveries
  than guaranteeing zero false positives
- Bonferroni is overly conservative and loses statistical power
- As shown in the teacher's notebook, FDR is preferred when
  comparing many groups

---

### What are the practical implications?

> ⚠️ **The correction methods did not save us from a misleading conclusion
> — the sample size did.**

The real problem is not multiple comparisons — it is that with
50+ million clicks per channel, even a **0.09% difference** produces
p ≈ 0.000. No correction method changes the fact that these differences
are statistically detectable but **practically meaningless**.

The key takeaways for budget allocation are:
- **CPA:** No channel is significantly more or less costly — confirmed
  by all three methods
- **Conversion Rate:** Channels differ statistically but the largest
  difference is only **1.04%** (Email vs Paid Ads) — not actionable
- **Recommendation so far:** Do not reallocate budget based on these
  findings alone — the differences do not justify the risk of disrupting
  a working marketing mix


## Part 4: Power Analysis & Sample Size Planning

### Step 7: Empirical Power Analysis

**What is statistical power?**
Power is the probability of detecting a real difference when it actually
exists. The standard target is **80% power (β = 0.20)**.

Low power means we might miss real differences (false negatives).
High power with tiny effect sizes means we detect meaningless differences.

**What we calculate here:**
1. Power curves — how much power do we have to detect CPA differences
   of 5%, 10%, 15%, 20% across different sample sizes?
2. Minimum sample size needed to achieve 80% power for each effect size
3. Assessment of whether our current data (~11,000 rows per channel)
   is adequate for detecting meaningful differences

This follows the empirical simulation approach from the teacher's
notebook (03_statistical_analysis.ipynb).

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Power Simulation
# ═══════════════════════════════════════════════════════════════════

rng_power = np.random.default_rng(42)

def empirical_power_cpa(true_diff_pct, base_cpa, n_samples,
                         n_sim=1000, alpha=0.05):
    """
    Simulate power to detect a CPA difference of true_diff_pct%.
    
    Parameters:
        true_diff_pct : float — true difference as % of base CPA (e.g. 0.10 = 10%)
        base_cpa      : float — baseline CPA mean
        n_samples     : int   — number of campaigns per channel
        n_sim         : int   — number of simulations
        alpha         : float — significance threshold
    Returns:
        float — proportion of simulations that correctly rejected H0
    """
    std_cpa   = base_cpa * 0.15   # 15% coefficient of variation
    mean_b    = base_cpa * (1 + true_diff_pct)
    rejections = 0

    for _ in range(n_sim):
        a = rng_power.normal(base_cpa, std_cpa, n_samples)
        b = rng_power.normal(mean_b,   std_cpa, n_samples)
        # Cap at 50% of base to avoid unrealistic values
        a = np.clip(a, base_cpa * 0.5, None)
        b = np.clip(b, base_cpa * 0.5, None)
        _, p = ttest_ind(a, b)
        if p < alpha:
            rejections += 1

    return rejections / n_sim

# ── Parameters ────────────────────────────────────────────────────
base_cpa     = df["Acquisition_Cost"].mean()
effect_sizes = [0.05, 0.10, 0.15, 0.20]   # 5%, 10%, 15%, 20% difference
sample_sizes = [30, 60, 90, 120, 180, 300, 500, 1000]

print(f"Base CPA: ${base_cpa:.2f}")
print(f"Simulating power for effect sizes: {[f'{e*100:.0f}%' for e in effect_sizes]}")
print(f"Sample sizes tested: {sample_sizes}")
print("\nRunning simulations (this may take ~30 seconds)...")

# ── Run simulations ───────────────────────────────────────────────
power_records = []
for eff in effect_sizes:
    for n in sample_sizes:
        pw = empirical_power_cpa(eff, base_cpa, n, n_sim=1000)
        power_records.append({
            "Effect Size":     f"{int(eff*100)}%",
            "Effect Size Pct": eff,
            "N Samples":       n,
            "Power":           pw,
        })

power_df = pd.DataFrame(power_records)
print("✅ Simulations complete!")

# ── Power curves plot ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
colors_pw = sns.color_palette("husl", len(effect_sizes))

for eff, color in zip(effect_sizes, colors_pw):
    subset = power_df[power_df["Effect Size Pct"] == eff]
    ax.plot(subset["N Samples"], subset["Power"],
            marker="o", label=f"{int(eff*100)}% difference",
            color=color, linewidth=2)

ax.axhline(0.80, color="red", linestyle="--",
           linewidth=1.5, label="80% power target")
ax.set_xlabel("Sample Size (campaigns per channel)")
ax.set_ylabel("Statistical Power")
ax.set_title("Power to Detect CPA Differences\nby Effect Size and Sample Size")
ax.legend(title="True CPA Difference")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("power_analysis_cpa.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved power_analysis_cpa.png")

# ── Power table ───────────────────────────────────────────────────
print("\n" + "=" * 65)
print("POWER TABLE — by Effect Size and Sample Size")
print("=" * 65)
pivot = power_df.pivot(index="N Samples",
                       columns="Effect Size",
                       values="Power").round(2)
print(pivot.to_string())

### Step 7b: Minimum Sample Size & Current Data Adequacy

Now we determine:
1. The **minimum number of campaigns** needed per channel to achieve
   80% power for each effect size
2. Whether our current dataset (~11,000 campaigns per channel) is
   **sufficient** to detect meaningful CPA differences

We also assess power for the observed differences from our actual
t-test results — i.e., given what we found, did we have enough data
to detect it if it were real?

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Min Sample Size & Data Adequacy
# ═══════════════════════════════════════════════════════════════════

target_power  = 0.80
current_n     = int(df.groupby("Campaign_Type").size().mean())

print("=" * 60)
print("MINIMUM SAMPLE SIZE FOR 80% POWER")
print("=" * 60)
print(f"{'Effect Size':<15} {'Min N Needed':>14} {'Current N':>12} {'Sufficient?':>12}")
print("-" * 55)

for eff in effect_sizes:
    subset = power_df[power_df["Effect Size Pct"] == eff].sort_values("N Samples")
    # Find first N where power >= 0.80
    sufficient_rows = subset[subset["Power"] >= target_power]
    if len(sufficient_rows) > 0:
        min_n = int(sufficient_rows.iloc[0]["N Samples"])
    else:
        min_n = ">1000"
    status = "✅ Sufficient" if isinstance(min_n, int) and current_n >= min_n else "❌ Insufficient"
    print(f"{int(eff*100):>3}% difference   {str(min_n):>14} {current_n:>12,} {status:>12}")

# ── Current data adequacy for observed differences ────────────────
print("\n" + "=" * 60)
print("POWER ASSESSMENT FOR OBSERVED CPA DIFFERENCES")
print("=" * 60)
print(f"(Using current N = {current_n:,} campaigns per channel)")
print()

for _, row in cpa_results.iterrows():
    obs_diff_pct = abs(row["% Difference"]) / 100
    if obs_diff_pct == 0:
        continue
    pw = empirical_power_cpa(obs_diff_pct, base_cpa,
                              current_n, n_sim=500)
    status = "✅ Adequate" if pw >= 0.80 else "❌ Inadequate"
    print(f"{row['Channel A']:>12} vs {row['Channel B']:<14} "
          f"diff={row['% Difference']:>5.2f}%  "
          f"power={pw:.2f}  {status}")

# ── Key conclusion ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("KEY CONCLUSION")
print("=" * 60)
print(f"Current sample size per channel: {current_n:,} campaigns")
print(f"This gives us >99% power to detect even a 5% CPA difference.")
print(f"Since we detected NO significant CPA differences, we can")
print(f"conclude with high confidence that channels truly perform")
print(f"equivalently on CPA — this is NOT a data adequacy problem.")

## Part 4 Observations: Power Analysis & Sample Size Planning

### Power Curves

The power curves show how quickly we gain the ability to detect
CPA differences as sample size increases:

| Effect Size | Min N for 80% Power | Interpretation |
|-------------|-------------------|----------------|
| 5% difference (~$19) | 180 campaigns | Achievable with ~6 months of data |
| 10% difference (~$38) | 60 campaigns | Achievable with ~2 months of data |
| 15% difference (~$57) | 30 campaigns | Achievable with ~1 month of data |
| 20% difference (~$75) | 30 campaigns | Achievable with ~1 month of data |

**Our current dataset (11,111 campaigns per channel) is sufficient
for all effect sizes** — we have far more data than needed.

---

### Is our data adequate?

With 11,111 campaigns per channel we have:
- **>99% power** to detect a 5% CPA difference
- **100% power** to detect differences of 10% or more

This means our null results from Part 2 are **highly reliable** —
we are not missing real differences due to insufficient data.

---

### Power Assessment for Observed Differences

The observed CPA differences between channels ranged from 0.10% to
1.81%. The power assessment reveals an interesting split:

**✅ Adequate power (power ≥ 0.80):**
- Email vs Social Media (diff = 1.81%, power = 1.00)
- Email vs Influencer (diff = 1.22%, power = 1.00)
- Influencer vs Paid Ads (diff = 0.97%, power = 1.00)
- Paid Ads vs Social Media (diff = 1.55%, power = 1.00)
- SEO vs Social Media (diff = 1.45%, power = 1.00)

**❌ Inadequate power (power < 0.80):**
- Email vs Paid Ads (diff = 0.26%, power = 0.24)
- Email vs SEO (diff = 0.37%, power = 0.47)
- Paid Ads vs SEO (diff = 0.10%, power = 0.08)

> **Interpretation:** The pairs with "inadequate power" are not
> failing because we have too little data — they are failing because
> the differences are so tiny (0.10%–0.37%) that even 11,111 samples
> cannot reliably detect them. This is a signal that these differences
> are **effectively zero**, not that we need more data.

---

### Key Conclusion

> Our analysis is **not limited by sample size**. With 11,111
> campaigns per channel, we have overwhelming power to detect any
> meaningful CPA difference (≥5%). The fact that no significant
> differences were found is strong evidence that the 5 channels
> are genuinely equivalent in cost efficiency.
>
> The 3 pairs showing inadequate power have differences so small
> (< 0.4%) that they would be **practically meaningless** even if
> we had infinite data to detect them reliably.


## Part 5: Business Recommendations

### Step 8: Synthesize Findings & Create Recommendations

We now translate our statistical findings into actionable business
recommendations for the CMO.

**Approach:**
1. Summarise statistically significant findings (FDR-corrected)
2. Calculate 95% bootstrap confidence intervals for CPA per channel
3. Rank channels using a composite score across all metrics
4. Propose a $500K monthly budget allocation
5. Write an executive memo with proper statistical caveats

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Significant Findings & Bootstrap CIs
# ═══════════════════════════════════════════════════════════════════

# ── Summary of FDR-significant findings ──────────────────────────
print("=" * 60)
print("STATISTICALLY SIGNIFICANT FINDINGS (FDR-corrected)")
print("=" * 60)
print("\nCPA Differences:")
print(f"  → 0 significant pairs after FDR correction")
print(f"  → Channels are statistically equivalent on CPA")

print("\nConversion Rate Differences:")
sig_fisher = fisher_df[fisher_df["significant_fdr"]].copy()
print(f"  → {len(sig_fisher)}/10 pairs significant after FDR correction")
print(f"  → Largest difference: "
      f"{sig_fisher['% Difference'].abs().max():.2f}% "
      f"(Email vs Paid Ads)")
print(f"  → Smallest difference: "
      f"{sig_fisher['% Difference'].abs().min():.2f}% "
      f"(Influencer vs SEO)")
print(f"  → All differences < 1.1% — statistically significant")
print(f"     but practically negligible")

# ── Bootstrap Confidence Intervals for CPA ───────────────────────
def bootstrap_ci(data, n_bootstrap=1000, ci_level=0.95):
    """Calculate bootstrap confidence interval for the mean."""
    rng_bs = np.random.default_rng(42)
    means  = [rng_bs.choice(data, size=len(data), replace=True).mean()
              for _ in range(n_bootstrap)]
    alpha  = (1 - ci_level) / 2
    return np.percentile(means, [alpha * 100, (1 - alpha) * 100])

print("\n" + "=" * 60)
print("95% BOOTSTRAP CONFIDENCE INTERVALS — CPA")
print("=" * 60)
print(f"{'Channel':<15} {'Mean CPA':>10} {'95% CI Lower':>14} {'95% CI Upper':>14}")
print("-" * 55)

ci_records = []
for ch in sorted(df["Campaign_Type"].unique()):
    vals = df[df["Campaign_Type"] == ch]["Acquisition_Cost"].dropna().values
    lo, hi = bootstrap_ci(vals)
    mean   = vals.mean()
    print(f"{ch:<15} ${mean:>9.2f}   ${lo:>12.2f}   ${hi:>12.2f}")
    ci_records.append({"Channel": ch, "Mean_CPA": mean,
                        "CI_Lower": lo, "CI_Upper": hi})

ci_df = pd.DataFrame(ci_records)

# ── CI Visualization ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors_ci = sns.color_palette("husl", len(ci_df))
for i, row in ci_df.iterrows():
    ax.barh(row["Channel"], row["Mean_CPA"], color=colors_ci[i],
            alpha=0.7, label=row["Channel"])
    ax.errorbar(row["Mean_CPA"], row["Channel"],
                xerr=[[row["Mean_CPA"] - row["CI_Lower"]],
                      [row["CI_Upper"] - row["Mean_CPA"]]],
                fmt="none", color="black", capsize=5, linewidth=2)
ax.set_xlabel("Average CPA ($)")
ax.set_title("Mean CPA by Channel with 95% Bootstrap Confidence Intervals")
plt.tight_layout()
plt.savefig("cpa_confidence_intervals.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved cpa_confidence_intervals.png")


### Step 8b: Channel Ranking & Budget Allocation

Our statistical analysis found **no significant differences** between
channels on CPA, ROAS or ROI. This has a direct implication for budget
allocation: **there is no statistical justification for strongly
favouring one channel over another.**

We present two allocation scenarios:

#### Scenario A — Equal Allocation (Primary Recommendation)
Since we cannot reject the null hypothesis that all channels perform
equivalently, the statistically honest recommendation is to allocate
the $500K budget **equally across all 5 channels** at $100K (20%) each.
Acting on small, non-significant differences would risk shifting budget
based on noise — exactly the costly mistake this analysis was designed
to prevent.

#### Scenario B — Modest Tilt (Secondary, if CMO requires differentiation)
If the CMO requires a differentiated allocation, we apply a **modest
±5% tilt** from equal share based on a composite ranking across all
available metrics (CPA, ROAS, ROI, Conversion Rate, Engagement Score).
This limits the downside risk of acting on unproven differences while
still reflecting marginal performance trends.

**Composite score methodology:**
- Rank each channel on: CPA (lower = better), ROAS, ROI, Conversion
  Rate, Engagement Score (all higher = better)
- Average the ranks into a composite score (lower = better overall)
- Constrain allocations to ±5% around the equal share of 20%

> ⚠️ **Critical caveat:** Scenario B is **not statistically justified**.
> All observed differences are negligible (Cohen's d < 0.02). Scenario B
> should only be adopted alongside a commitment to re-evaluate after
> 90 days of controlled experimentation with pre-specified minimum
> detectable effects.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Budget Allocation
# ═══════════════════════════════════════════════════════════════════

TOTAL_BUDGET = 500_000
N_CHANNELS   = 5

# ── Build composite ranking ───────────────────────────────────────
rank_df = agg[["Campaign_Type", "Avg_CPA", "Avg_ROAS",
               "Avg_ROI", "Avg_Conv_Rate", "Avg_Engagement"]].copy()

rank_df["rank_CPA"]      = rank_df["Avg_CPA"].rank(ascending=True)
rank_df["rank_ROAS"]     = rank_df["Avg_ROAS"].rank(ascending=False)
rank_df["rank_ROI"]      = rank_df["Avg_ROI"].rank(ascending=False)
rank_df["rank_ConvRate"] = rank_df["Avg_Conv_Rate"].rank(ascending=False)
rank_df["rank_Engage"]   = rank_df["Avg_Engagement"].rank(ascending=False)

rank_cols = ["rank_CPA", "rank_ROAS", "rank_ROI",
             "rank_ConvRate", "rank_Engage"]
rank_df["Composite_Score"] = rank_df[rank_cols].mean(axis=1)
rank_df["Weight"] = (N_CHANNELS + 1) - rank_df["Composite_Score"]
rank_df["Weight"] = rank_df["Weight"] / rank_df["Weight"].sum()
rank_df = rank_df.sort_values("Composite_Score")

# ── Scenario A: Equal allocation ──────────────────────────────────
equal_alloc = TOTAL_BUDGET / N_CHANNELS
rank_df["ScenarioA_$"]   = equal_alloc
rank_df["ScenarioA_Pct"] = 1 / N_CHANNELS

# ── Scenario B: Modest tilt ±5% from equal ────────────────────────
rank_df["ScenarioB_Pct"] = rank_df["Weight"].clip(
    lower=1/N_CHANNELS - 0.05,
    upper=1/N_CHANNELS + 0.05
)
rank_df["ScenarioB_Pct"] = rank_df["ScenarioB_Pct"] / rank_df["ScenarioB_Pct"].sum()
rank_df["ScenarioB_$"]   = (rank_df["ScenarioB_Pct"] * TOTAL_BUDGET).round(-2)

# ── Print tables ──────────────────────────────────────────────────
print("=" * 60)
print("SCENARIO A — EQUAL ALLOCATION (Primary Recommendation)")
print("Justification: No statistically significant CPA/ROAS differences")
print("=" * 60)
print(f"{'Channel':<15} {'Allocation $':>14} {'Allocation %':>14}")
print("-" * 45)
for _, row in rank_df.iterrows():
    print(f"{row['Campaign_Type']:<15} ${equal_alloc:>12,.0f}          20.0%")
print("-" * 45)
print(f"{'TOTAL':<15} ${TOTAL_BUDGET:>12,.0f}         100.0%")

print("\n" + "=" * 60)
print("SCENARIO B — MODEST TILT ±5% (Secondary, not statistically")
print("justified — only if CMO requires differentiation)")
print("=" * 60)
print(f"{'Channel':<15} {'Allocation $':>14} {'Allocation %':>14}")
print("-" * 45)
for _, row in rank_df.iterrows():
    print(f"{row['Campaign_Type']:<15} ${row['ScenarioB_$']:>12,.0f} "
          f"{row['ScenarioB_Pct']*100:>13.1f}%")
print("-" * 45)
print(f"{'TOTAL':<15} ${rank_df['ScenarioB_$'].sum():>12,.0f}")

# ── Visualization ─────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Budget Allocation Scenarios", fontsize=14, fontweight="bold")

channels_sorted = rank_df["Campaign_Type"].tolist()
colors_alloc    = sns.color_palette("husl", N_CHANNELS)

# Scenario A
bars1 = ax1.barh(channels_sorted, rank_df["ScenarioA_$"],
                 color=colors_alloc, alpha=0.8)
ax1.set_xlabel("Budget ($)")
ax1.set_title("Scenario A: Equal Allocation\n[PRIMARY] Statistically justified")
for bar in bars1:
    ax1.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
             "$100K (20.0%)", va="center", fontsize=9)
ax1.set_xlim(0, 160000)
ax1.axvline(equal_alloc, color="red", linestyle="--",
            linewidth=1, alpha=0.5, label="Equal share")

# Scenario B
bars2 = ax2.barh(channels_sorted, rank_df["ScenarioB_$"],
                 color=colors_alloc, alpha=0.8)
ax2.set_xlabel("Budget ($)")
ax2.set_title("Scenario B: Modest Tilt +/-5%\n[SECONDARY] Not statistically justified")
for bar, val, pct in zip(bars2, rank_df["ScenarioB_$"],
                          rank_df["ScenarioB_Pct"] * 100):
    ax2.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
             f"${val/1000:.0f}K ({pct:.1f}%)", va="center", fontsize=9)
ax2.set_xlim(0, 160000)
ax2.axvline(equal_alloc, color="red", linestyle="--",
            linewidth=1, alpha=0.5, label="Equal share")

plt.tight_layout()
plt.savefig("budget_allocation.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved budget_allocation.png")

### Step 8c: Executive Memo

Writing a professional executive memo summarising all findings,
recommendations and statistical caveats for the CMO.
The memo is saved as executive_memo.md.

In [ ]:
# ===================================================================
# Generate Executive Memo
# ===================================================================

memo = """# EXECUTIVE MEMO

**To:** Chief Marketing Officer
**From:** Marketing Analytics Team
**Date:** {date}
**Re:** Statistical Analysis of Marketing Channel Performance
**Dataset:** Nykaa Marketing Campaign Performance Dataset (Kaggle)
**Period Analysed:** July 2024 - June 2025 (55,555 campaigns, 5 channels)

---

## Executive Summary

A rigorous statistical analysis of 55,555 marketing campaigns across
5 channels (Email, Influencer, Paid Ads, SEO, Social Media) found
**no statistically significant differences in CPA or ROAS** between
any channel pair after multiple comparisons correction.

Conversion rate differences were statistically detectable but
practically negligible (largest difference: 1.04%).

**Primary recommendation: maintain an equal $100K allocation per
channel.** There is no statistical evidence to justify shifting
budget away from any channel at this time.

---

## Key Findings

### 1. Channel Performance Overview

| Channel | Avg CPA | Avg ROAS | Avg ROI | Avg Conv Rate |
|---------|---------|----------|---------|---------------|
| Social Media | $373 | 3.75 | 2.75 | 21.92% |
| Influencer | $375 | 3.70 | 2.70 | 22.01% |
| Paid Ads | $379 | 3.72 | 2.72 | 21.81% |
| SEO | $379 | 3.71 | 2.71 | 22.05% |
| Email | $380 | 3.68 | 2.68 | 22.08% |

Social Media ranks first on CPA, ROAS and ROI.
Email ranks last on those same metrics despite the highest
conversion rate, suggesting higher cost per click rather
than lower conversion effectiveness.

### 2. Statistical Test Results

**CPA — Independent t-tests (10 pairwise comparisons):**
- Significant before correction: 0/10
- Significant after Bonferroni: 0/10
- Significant after FDR (BH): 0/10
- Largest difference: $6.87 (Email vs Social Media, 1.81%)
- All Cohen's d: negligible (< 0.02)
- Verdict: channels are statistically equivalent on CPA

**Conversion Rate — Fisher's Exact Test (10 pairwise comparisons):**
- Significant before correction: 10/10
- Significant after Bonferroni: 9/10
- Significant after FDR (BH): 10/10
- Largest difference: 1.04% (Email vs Paid Ads)
- Verdict: statistically significant but practically negligible

### 3. Data Adequacy

With 11,111 campaigns per channel our analysis has:
- >99% power to detect a 5% CPA difference (~$19)
- 100% power to detect a 10% CPA difference (~$38)

The null results on CPA are therefore not due to insufficient data.
We can conclude with high confidence that channels genuinely
perform equivalently on cost efficiency.

### 4. Confidence Intervals for CPA

| Channel | Mean CPA | 95% CI Lower | 95% CI Upper |
|---------|---------|-------------|-------------|
| Social Media | $373 | $364 | $383 |
| Influencer | $375 | $365 | $386 |
| Paid Ads | $379 | $369 | $389 |
| SEO | $379 | $369 | $389 |
| Email | $380 | $370 | $391 |

All confidence intervals heavily overlap, confirming no channel
is reliably cheaper than any other.

---

## Budget Allocation Recommendations

### Scenario A — Equal Allocation (PRIMARY RECOMMENDATION)

| Channel | Budget | Share |
|---------|--------|-------|
| Social Media | $100,000 | 20.0% |
| Influencer | $100,000 | 20.0% |
| Paid Ads | $100,000 | 20.0% |
| SEO | $100,000 | 20.0% |
| Email | $100,000 | 20.0% |
| **TOTAL** | **$500,000** | **100%** |

Justification: no statistically significant differences detected
on any cost or return metric. Reallocating based on non-significant
differences risks shifting budget based on noise.

### Scenario B — Modest Tilt (SECONDARY, requires justification)

| Channel | Budget | Share |
|---------|--------|-------|
| Social Media | $126,700 | 25.3% |
| Influencer | $108,100 | 21.6% |
| Paid Ads | $94,600 | 18.9% |
| SEO | $94,600 | 18.9% |
| Email | $76,000 | 15.2% |
| **TOTAL** | **$500,000** | **100%** |

Justification: if differentiation is required, a modest +-5%
tilt based on composite ranking limits downside risk. This
scenario is NOT statistically justified and should only be
adopted alongside a 90-day controlled experiment.

---

## Strategic Actions

1. **Do not make large reallocations** based on current data.
   The statistical evidence does not support it.

2. **Focus on campaign-level optimisation.** CPA standard
   deviations ($520-$566) far exceed between-channel differences
   ($7), meaning individual campaign quality matters far more
   than channel selection.

3. **Run controlled A/B experiments.** Before any major
   reallocation, run 90-day experiments with pre-specified
   minimum detectable effect of 5% CPA difference (~$19).

4. **Investigate high-CPA outliers.** Some campaigns show CPA
   above $5,000 vs a median of ~$200. Eliminating these outliers
   could reduce costs more than any channel reallocation.

5. **Review Email channel.** While not significantly worse,
   Email consistently ranks last on CPA and ROAS. Audit
   targeting, creative quality and list hygiene before the
   next budget cycle.

---

## Statistical Caveats

1. **Dataset:** The Nykaa dataset is described as real-world
   inspired — findings should be validated against live
   platform data before acting.

2. **Multiple comparisons correction:** All p-values corrected
   using Benjamini-Hochberg FDR (alpha=0.05) across 20
   simultaneous comparisons.

3. **Statistical vs practical significance:** Fisher's exact
   tests detected conversion rate differences as small as 0.09%
   as significant. P-values alone should never drive decisions.

4. **High within-channel variance:** CPA std dev of $520-$566
   per channel dwarfs between-channel differences. External
   factors (seasonality, campaign quality, targeting) likely
   explain more variance than channel choice.

5. **Observational analysis:** No causal inference can be drawn.
   Correlation between channel and performance does not guarantee
   that reallocating budget will replicate the same results.

6. **Power analysis assumptions:** Simulations assumed normal
   distribution with 15% coefficient of variation. Actual CPA
   distributions are right-skewed which may affect estimates.

---

## Next Steps

1. Validate findings against live Google Analytics / platform data
2. Run 90-day controlled A/B experiments with pre-specified MDE
3. Investigate high-CPA outlier campaigns (CPA > $5,000)
4. Conduct campaign-level analysis within each channel
5. Re-run this analysis quarterly to track divergence over time

---
*Analysis: Python (pandas, scipy, numpy, matplotlib, seaborn).
Full code and outputs available in the accompanying notebook.*
""".format(date=pd.Timestamp.today().strftime("%B %d, %Y"))

with open("executive_memo.md", "w") as f:
    f.write(memo)

print("✅ Saved executive_memo.md")
print("\nPreview (first 300 chars):")
print(memo[:300])

# LAB DELIVERABLES CHECKLIST

## Files to Submit

### Code & Data
- [x] `data_exploration.ipynb` — main notebook with all code
- [x] `marketing_data.csv` — cleaned dataset

### Visualizations (PNG files)
- [x] `group_metrics_overview.png` — bar charts of all KPIs
- [x] `group_distributions.png` — box plots of CPA, ROAS, Conv Rate
- [x] `cpa_comparison_heatmap.png` — p-value heatmap for CPA
- [x] `roas_comparison_heatmap.png` — p-value heatmap for ROAS
- [x] `rate_comparison.png` — conversion rate bar chart
- [x] `correction_comparison.png` — effect of multiple comparisons correction
- [x] `power_analysis_cpa.png` — power curves
- [x] `cpa_confidence_intervals.png` — bootstrap CIs
- [x] `budget_allocation.png` — two allocation scenarios

### Written Deliverables
- [x] `executive_memo.md` — full memo with findings and caveats

---

## Lab Success Criteria Check

### Part 1 — Dataset & Exploration
- [x] Dataset documented (name, source, URL, why chosen)
- [x] Data loaded, cleaned and saved as CSV
- [x] Key metrics calculated: CPA, ROAS, Conv Rate, CTR
- [x] Group summary table printed
- [x] Bar charts and distribution plots created

### Part 2 — Statistical Tests
- [x] Pairwise t-tests run for CPA and ROAS (10 pairs each)
- [x] Cohen's d calculated and interpreted for all pairs
- [x] Fisher's exact test run for conversion rates (10 pairs)
- [x] P-value heatmaps created
- [x] Conversion rate bar chart created

### Part 3 — Multiple Comparisons Correction
- [x] Expected false positives calculated (20 x 0.05 = 1.0)
- [x] Bonferroni correction applied to CPA and conversion rate
- [x] BH-FDR correction applied to CPA and conversion rate
- [x] Summary table comparing all three methods
- [x] Correction comparison bar chart created

### Part 4 — Power Analysis
- [x] Empirical power simulation function written
- [x] Power curves for 5%, 10%, 15%, 20% effect sizes
- [x] Minimum sample size table for 80% power
- [x] Current data adequacy assessed for all channel pairs
- [x] Power curve plot saved

### Part 5 — Business Recommendations
- [x] FDR-significant findings summarised
- [x] Bootstrap 95% CIs calculated for CPA per channel
- [x] Composite ranking table produced
- [x] Two budget allocation scenarios presented
- [x] Equal allocation justified by statistical findings
- [x] Executive memo written with all required sections

---

## Markdown Observations Cells Check
- [x] Part 1 observations (4 key insight questions answered)
- [x] Part 2 observations (t-test + Fisher's findings)
- [x] Part 3 observations (correction methods compared)
- [x] Part 4 observations (power and data adequacy)

In [ ]:

# ===================================================================
# Generate README.md
# ===================================================================

readme = """# Lab: Marketing Channel ROI Comparison
## Statistical Analysis for Budget Allocation

---

## Overview

This lab performs a statistically rigorous analysis of marketing
channel performance to guide the allocation of a $500K monthly
budget across 5 channels. The analysis follows a full statistical
pipeline: data exploration, pairwise hypothesis testing, multiple
comparisons correction, power analysis, and business recommendations.

---

## Dataset

**Name:** Nykaa Marketing Campaign Performance Dataset
**Source:** Kaggle
**File:** nykaa_campaign_data.csv
**Size:** 55,555 campaigns x 16 columns
**Period:** July 2024 - June 2025
**Channels:** Email, Influencer, Paid Ads, SEO, Social Media (~11,000
campaigns per channel)

A structured, real-world inspired e-commerce marketing campaign
dataset designed for ROI analysis, performance optimization, and
predictive modeling.

---

## Key Findings

1. **No significant CPA differences:** 0/10 channel pairs showed
   statistically significant CPA differences after Bonferroni and
   FDR correction. All Cohen's d values were negligible (< 0.02).

2. **Negligible conversion rate differences:** All 10 channel pairs
   showed statistically significant conversion rate differences after
   FDR correction, but the largest difference was only 1.04% —
   practically meaningless.

3. **Data is not the limiting factor:** With 11,111 campaigns per
   channel, the analysis has >99% power to detect a 5% CPA
   difference. Null results reflect genuine channel equivalence,
   not insufficient data.

4. **Within-channel variance dominates:** CPA standard deviations
   of $520-$566 per channel dwarf between-channel differences of
   just $7, suggesting campaign-level quality matters far more
   than channel selection.

5. **Primary recommendation: equal allocation.** $100K per channel
   is the statistically defensible recommendation. A secondary
   modest tilt scenario (Social Media 25.3%, Email 15.2%) is
   provided if differentiation is required, with appropriate caveats.

---

## How to Run

1. Install dependencies:
   ```
   pip install numpy pandas matplotlib seaborn scipy
   ```

2. Place `nykaa_campaign_data.csv` in the same folder as the notebook

3. Open and run `data_exploration.ipynb` from top to bottom
   - All cells are sequential and must be run in order
   - Power analysis cells (~30 seconds to run)

4. All output files are saved automatically to the same folder

---

## File Structure

```
LAB702/
├── data_exploration.ipynb        # Main analysis notebook
├── nykaa_campaign_data.csv       # Raw dataset
├── marketing_data.csv            # Cleaned dataset
├── executive_memo.md             # Business recommendations memo
├── README.md                     # This file
├── group_metrics_overview.png    # KPI bar charts
├── group_distributions.png       # Box plots by channel
├── cpa_comparison_heatmap.png    # P-value heatmap (CPA)
├── roas_comparison_heatmap.png   # P-value heatmap (ROAS)
├── rate_comparison.png           # Conversion rate comparison
├── correction_comparison.png     # Multiple comparisons correction
├── power_analysis_cpa.png        # Power curves
├── cpa_confidence_intervals.png  # Bootstrap confidence intervals
└── budget_allocation.png         # Budget allocation scenarios
```

---

## Assumptions & Notes

- **Derived metrics:** Spend = Acquisition_Cost x Conversions;
  ROAS = Revenue / Spend; Conv_Rate = Conversions / Clicks;
  CTR = Clicks / Impressions
- **Statistical threshold:** alpha = 0.05 throughout
- **Multiple comparisons:** Benjamini-Hochberg FDR applied to all
  pairwise comparisons (20 total)
- **Power simulation:** 1,000 simulations per condition, normal
  distribution assumed with 15% coefficient of variation
- **Bootstrap CIs:** 1,000 resamples, 95% confidence level
- **Budget constraints:** Scenario B allocations constrained to
  +/-5% around equal share (20%) per channel

---

## Statistical Methods Used

| Method | Purpose |
|--------|---------|
| Independent t-test | Compare CPA and ROAS between channel pairs |
| Fisher's exact test | Compare binary conversion outcomes |
| Cohen's d | Measure practical effect size |
| Bonferroni correction | Control family-wise error rate |
| Benjamini-Hochberg FDR | Control false discovery rate |
| Empirical power simulation | Assess data adequacy |
| Bootstrap confidence intervals | Quantify CPA uncertainty |
"""

with open("README.md", "w") as f:
    f.write(readme)

print("✅ Saved README.md")
print("\nPreview (first 300 chars):")
print(readme[:300])